# **Observability in CrewAI**

In this notebook we'll explore how to integrate Langfuse with CrewAI via OpenTelemetry using OpenLit.

## In the context of CrewAI **observability** essentially means being able to see and understand what happened inside a Crew execution, rather than only seeing the final output.

CrewAI's Langfuse integration uses OpenTelemetry/OpenLit to capture traces of the CrewAI execution and send them to Langfuse.


## What does it let you observe?

For a multi-agent workflow, you can inspect things such as:


> Which agent executed which task

> LLM calls — prompts, responses, model usage

> Tool calls — which tool was invoked and when

> Execution flow — how agents/tasks progressed

> Latency — where time was spent

> Token usage / cost

> Errors and failures

> Intermediate steps, rather than just the final answer

This is particularly important because an agentic system is a multi-step, non-deterministic process. A bad final answer may actually be caused by an earlier tool call, poor intermediate reasoning, wrong agent decision, or excessive LLM calls.

Using observability, you can **debug**, **optimize**, and **monitor** the actual behavior of the crew. CrewAI's documentation explicitly positions the Langfuse integration for tracing CrewAI applications to improve observability and debugging.




### One important distinction, **Observability ≠ evaluation**.

### Observability tells you: "What happened inside my Crew, and why?"

### Evaluation asks: "Was the result actually good?"

### And this distinction is important for the CrewAI callback/checkpointing work we've been discussing: callbacks can capture events and metrics inside your application, while an observability platform such as Langfuse provides a much more complete trace of the execution and a UI for analyzing it.

### So, in one sentence: **CrewAI observability is the ability to trace, monitor, and understand the execution of agents, tasks, LLM calls, tools, and their interactions throughout a Crew workflow.**

## What is Langfuse?

### **Langfuse** is an **open-source LLM engineering platform**.

### It provides tracing and monitoring capabilities for LLM applications, helping developers debug, analyze, and optimize their AI systems.

### Langfuse integrates with various tools and frameworks via native integrations, OpenTelemetry, and APIs/SDKs.


## Singnup for your account:

https://langfuse.com/cloud


Create a new project and generate LANGFUSE_SECRET_KEY and LANGFUSE_PUBLIC_KEY and note down the LANGFUSE_BASE_URL (also known as the LANGFUSE_HOST).

## Install necessary Libraries

In [ ]:
!pip install -q crewai crewai_tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 836.4/836.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 27.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-instrumentation-fl

In [ ]:
!pip install langfuse openlit

  Using cached opentelemetry_exporter_otlp_proto_grpc-1.44.0-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 695.4/695.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.3/646.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.42.1
    Unin

In [ ]:
import crewai, crewai_tools
print(crewai.__version__)
print(crewai_tools.__version__)


1.15.17
1.15.17


# Set Up Environment Variables

Set your Langfuse API keys and configure OpenTelemetry export settings to send traces to Langfuse.

Please refer to the Langfuse OpenTelemetry Docs [https://langfuse.com/docs/opentelemetry/get-started] for more information on the Langfuse OpenTelemetry endpoint /api/public/otel and authentication.

In [ ]:
from google.colab import userdata
import os
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_BASE_URL"] ="https://cloud.langfuse.com" # also called the HOST

## Initialize the Langfuse client

With the environment variables set, we can now initialize the Langfuse client. get_client() initializes the Langfuse client using the credentials provided in the environment variables.

In [ ]:
from langfuse import get_client

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

Langfuse client is authenticated and ready!


## Initialize OpenLit

Initialize the OpenLit OpenTelemetry instrumentation SDK to start capturing OpenTelemetry traces.

In [ ]:
import openlit

openlit.init()

## Import Dependencies

In [ ]:
from crewai import Agent, Task, Crew, LLM
from crewai_tools import SerperDevTool

## Create the LLM Object

In [ ]:
# OPENROUTER hosted LLMs
llm = LLM(
    model="openrouter/openai/gpt-oss-120b",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

# Create a CrewAI Application

In [ ]:
# Define a tool
web_search_tool = SerperDevTool()

writer = Agent(
        role="Writer",
        goal="You make math engaging and understandable for young children through poetry",
        backstory="You're an expert in writing haikus but you know nothing of math.",
        llm=llm,
        tools=[web_search_tool],
    )

task = Task(description=("What is {multiplication}?"),
            expected_output=("Compose a haiku that includes the answer."),
            agent=writer)

crew = Crew(
  agents=[writer],
  tasks=[task],
  share_crew=False
)

## Execute the Crew

In [ ]:
result = await crew.kickoff_async()
result

{
    "body": "",
    "severity_number": null,
    "severity_text": null,
    "attributes": {
        "gen_ai.operation.name": "chat",
        "gen_ai.request.model": "openai/gpt-oss-120b",
        "gen_ai.response.model": "openai/gpt-oss-120b",
        "server.address": "openrouter.ai",
        "server.port": 443,
        "gen_ai.input.messages": [
            {
                "role": "system",
                "parts": [
                    {
                        "type": "text",
                        "content": "You are Writer. You're an expert in writing haikus but you know nothing of math.\nYour personal goal is: You make math engaging and understandable for young children through poetry"
                    }
                ]
            },
            {
                "role": "user",
                "parts": [
                    {
                        "type": "text",
                        "content": "\nCurrent Task: What is {multiplication}?\n\nThis is the expected c

CrewOutput(raw='Multiplication,  \nadding same groups, one by one—  \nnumbers grow like beans.', pydantic=None, json_dict=None, tasks_output=[TaskOutput(description='What is {multiplication}?', name='What is {multiplication}?', expected_output='Compose a haiku that includes the answer.', summary='What is {multiplication}?...', raw='Multiplication,  \nadding same groups, one by one—  \nnumbers grow like beans.', pydantic=None, json_dict=None, agent='Writer', output_format=<OutputFormat.RAW: 'raw'>, messages=[{'role': 'system', 'content': "You are Writer. You're an expert in writing haikus but you know nothing of math.\nYour personal goal is: You make math engaging and understandable for young children through poetry"}, {'role': 'user', 'content': '\nCurrent Task: What is {multiplication}?\n\nThis is the expected criteria for your final answer: Compose a haiku that includes the answer.\nyou MUST return the actual complete content as the final answer, not a summary.'}, {'role': 'assistant

## See Traces in Langfuse

After running the agent, you can view the traces generated by your CrewAI application in Langfuse.

You should see detailed steps of the LLM interactions, which can help you debug and optimize your AI agent.